In [57]:
import torch
import torchmetrics
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import ToTensor

In [58]:
toTensor = T.Compose([
	T.ToImage(), 
	# T.Resize((28, 28)),
	T.ToDtype(torch.float32, scale=True)
])

train_and_valid_data = torchvision.datasets.FashionMNIST(
	root='./data', 
	train=True, 
	download=True,
	transform=ToTensor()
)

test_data = torchvision.datasets.FashionMNIST(
	root='./data', 
	train=False, 
	download=True,
	transform=ToTensor()
	)

train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [0.9, 0.1])

c:\Users\user\miniconda3\envs\torch\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [7]:
print(len(train_data), len(valid_data), len(test_data))

54000 6000 10000


In [9]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [10]:
# import torchvision.transforms.v2 as T

# # x = T.ToTensor()
# # x(img)

# transforms = T.Compose([
# 	T.ToTensor(),
# 	# T.RandomRotation((-90,90)),
# 	# T.Resize((100,100))
# ])

# plt.imshow(transforms(img).squeeze(0))

In [11]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cpu'

In [17]:
def train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
	history = {
		'loss' : [],
		'train_metric' : [],
		'valid_metric' : [],
	}
	for epoch in range(n_epochs):
		# Training 
		total_loss = 0
		metric.reset()
		for X_batch, y_batch in train_loader:
			X_batch, y_batch = X_batch.to(device), y_batch.to(device)
			model.train()
			y_pred = model(X_batch)
			loss = criterion(y_pred, y_batch)
			total_loss += loss.item()
			loss.backward()
			optimizer.step()
			# for group in optimizer.param_groups:
			# 	for p in group['params']:
			# 		if p.grad is not None:
			# 			print(f'\t{p.grad.max()}') 
			optimizer.zero_grad()
			metric.update(y_pred, y_batch)
		
		avg_loss = total_loss / len(train_loader)
		history['loss'].append(avg_loss)

		avg_metric_train = metric.compute().item()
		history['train_metric'].append(avg_metric_train)

		# Evaluation 
		model.eval()
		metric.reset()
		with torch.no_grad():
			for X_batch, y_batch in valid_loader:
				X_batch, y_batch = X_batch.to(device), y_batch.to(device)
				y_pred = model(X_batch)
				metric.update(y_pred, y_batch)

		avg_metric_valid = metric.compute().item()
		history['valid_metric'].append(avg_metric_valid)

		print(
			f'Epoch: {epoch+1}/{n_epochs}, '
			+f'Loss: {round(avg_loss,3)}, '
			+f'Train Metric: {round(avg_metric_train,3)}, ' 
			+f'Valid Metric: {round(avg_metric_valid,3)}'
		)

		# if epoch>=2:
		# 	break
	return history

def plot_history(history, n_epochs, metric):
    plt.plot(np.arange(n_epochs) + 1, history['train_metric'], linestyle='--', color='r', marker='.', label='Train')
    plt.plot(np.arange(n_epochs) + 1, history['valid_metric'], linestyle='--', color='b', marker='.', label='Valid')
    plt.legend()
    plt.grid()
    plt.xlabel('Epochs')
    plt.ylabel(f'{metric.__class__.__name__}')
    plt.show()

In [33]:
X_batch, y_batch = next(iter(train_loader))

In [ ]:
model = nn.Sequential(
	nn.Flatten(),
	nn.Linear(in_features=784, out_features=50),
	nn.Sigmoid(),
	nn.Linear(in_features=50, out_features=30),
	nn.Sigmoid(),
	nn.Linear(in_features=30, out_features=10),
)
criterion = nn.CrossEntropyLoss()
y_pred = model(X_batch)


In [37]:
y_pred[0]

tensor([-0.3911,  0.2394, -0.1231, -0.0040,  0.0256,  0.2727, -0.0406,  0.3605,
        -0.0359,  0.0342], grad_fn=<SelectBackward0>)

In [42]:
y_pred[0].softmax(0)

tensor([0.0640, 0.1203, 0.0837, 0.0943, 0.0971, 0.1244, 0.0909, 0.1358, 0.0914,
        0.0980], grad_fn=<SoftmaxBackward0>)

In [53]:
np.exp(0.2394) / np.exp(y_pred[0].detach().numpy()).sum()

np.float64(0.12031001218092091)

In [54]:
X_batch[0]

tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0392, 0.2784, 0.5490, 0.7922, 0.7137, 0.6039, 0.5686, 0.7020,
          0.7647, 0.3647, 0.1882, 0.0275, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2745, 0.4745,
          0.5216, 0.4745, 0.3451, 0.4471, 0.7961, 0.9137, 1.0000, 0.7843,
          0.4118, 0.3490, 0.4902, 0.5098, 0.4078, 0.1529, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4471, 0.4863, 0.4196,
          0.3725, 0.3647, 0.3647, 0.3137, 0.2157, 0.1647, 0.1686, 0.1961,
          0.2784, 0.3490, 0.3333, 0.3255, 0.4039, 0.5725, 0.2510, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.1176, 0.4941, 0.3804, 0.3725,
          0.3922, 0.3765, 0.3882, 0.3765, 0.3569, 0.3451, 0.3608, 0.3333,
          0.3412, 0.3569, 0.3569, 0.3490, 0.3255, 0.3608,